# KKBox Churn Prediction — Model Training

**Pipeline:**
- Data: `kkbox_gold.features_train` trên BigQuery (cutoff 2017-01-01)
- Model: XGBoost binary classification
- Tracking: MLflow
- Split: Out-of-time theo `registration_init_time` (cutoff 2016-06-01)

## 0. Install dependencies

In [ ]:
!pip install -q pandas-gbq xgboost scikit-learn mlflow google-cloud-bigquery pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.6/887.6 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

## 1. Authenticate GCP

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'kkbox-churn-prediction-493716'
print('Authenticated!')

Authenticated!


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
import time

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

In [ ]:
SPLIT_DATE = pd.Timestamp("2016-06-01")

DROP_COLS = ["msno", "registration_init_time", "latest_expire_date", "event_timestamp"]

NUM_COLS = [
    "total_transactions",
    "total_amount_paid",
    "avg_amount_paid",
    "auto_renew_count",
    "cancel_count",
    "total_log_days",
    "total_secs",
    "avg_daily_secs",
    "total_num_25",
    "total_num_50",
    "total_num_75",
    "total_num_985",
    "total_num_100",
    "total_num_unq",
]

GENDER_MAP = {"male": 0, "female": 1, "unknown": 2}

FEATURE_COLS = [
    "city",
    "bd",
    "gender",
    "registered_via",
    "total_transactions",
    "total_amount_paid",
    "avg_amount_paid",
    "auto_renew_count",
    "cancel_count",
    "total_log_days",
    "total_secs",
    "avg_daily_secs",
    "total_num_25",
    "total_num_50",
    "total_num_75",
    "total_num_985",
    "total_num_100",
    "total_num_unq",
]

TARGET_COL = "is_churn"

## Load data từ BigQuery

In [ ]:
import pandas_gbq
import pandas as pd

print('Loading data from BigQuery...')
df = pandas_gbq.read_gbq(
    'SELECT * FROM `kkbox-churn-prediction-493716.kkbox_gold.features_train`',
    project_id=PROJECT_ID
)

print(f'Shape: {df.shape}')
print(f'Churn rate: {df["is_churn"].mean():.2%}')
df.head()

Loading data from BigQuery...
Downloading: 100%|██████████|
Shape: (1082190, 23)
Churn rate: 10.05%


,msno,is_churn,city,bd,gender,registered_via,registration_init_time,total_transactions,total_amount_paid,avg_amount_paid,...,total_log_days,total_secs,avg_daily_secs,total_num_25,total_num_50,total_num_75,total_num_985,total_num_100,total_num_unq,event_timestamp
0,3UC9s3tSv/Z3huV/+aQFu1ADiMgDaQWDpygKfDPp9aM=,0,<NA>,<NA>,None,<NA>,NaT,<NA>,<NA>,NaN,...,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2016-12-31 00:00:00+00:00
1,fKj5wVWOKCT+pNFA9pSfLVmNJW6klOYyKPwcfYYSuHQ=,0,<NA>,<NA>,None,<NA>,NaT,<NA>,<NA>,NaN,...,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2016-12-31 00:00:00+00:00
2,4qVpLjS0rB+xVAOF2J4vbPIbZ7qEBmGMwPzT6V/6bcA=,0,<NA>,<NA>,None,<NA>,NaT,<NA>,<NA>,NaN,...,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2016-12-31 00:00:00+00:00
3,0jkN7dbzCAICtIbJiHOMl1eH+RkPA+M7cYY7mwSzClw=,1,<NA>,<NA>,None,<NA>,NaT,<NA>,<NA>,NaN,...,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2016-12-31 00:00:00+00:00
4,x4ArjvWk6oddhKjdtyDgZVc5s8nekY7YAglLYYKEyNM=,0,<NA>,<NA>,None,<NA>,NaT,<NA>,<NA>,NaN,...,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2016-12-31 00:00:00+00:00


# Preprocess

In [ ]:
def preprocess(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
  """Return processed DataFrame and the preprocessing config needed for inference."""
  df = df.copy()

  # Compute fill values from training data before dropping anything
  bd_median = float(df["bd"].median())
  city_mode = int(df["city"].mode()[0])
  registered_via_mode = int(df["registered_via"].mode()[0])

  preprocessing_config = {
      "drop_cols": DROP_COLS,
      "num_cols_fill_zero": NUM_COLS,
      "bd_median": bd_median,
      "city_mode": city_mode,
      "registered_via_mode": registered_via_mode,
      "gender_map": GENDER_MAP,
  }

  df[NUM_COLS] = df[NUM_COLS].fillna(0)
  df["gender"] = df["gender"].fillna("unknown")
  df["bd"] = df["bd"].fillna(bd_median)
  df["city"] = df["city"].fillna(city_mode)
  df["registered_via"] = df["registered_via"].fillna(registered_via_mode)
  df["gender"] = df["gender"].map(GENDER_MAP).fillna(2).astype(int)

  remaining_nulls = df[FEATURE_COLS + [TARGET_COL]].isnull().sum().sum()
  log.info("Preprocessing done — remaining nulls: %d", remaining_nulls)

  return df, preprocessing_config


# ── Split ─────────────────────────────────────────────────────────────────────

def split(df: pd.DataFrame, reg_time: pd.Series) -> tuple:
  train_mask = reg_time < SPLIT_DATE
  test_mask = reg_time >= SPLIT_DATE

  X_train = df.loc[train_mask, FEATURE_COLS]
  y_train = df.loc[train_mask, TARGET_COL]
  X_test = df.loc[test_mask, FEATURE_COLS]
  y_test = df.loc[test_mask, TARGET_COL]

  log.info(
      "Train: %d rows (%.2f%% churn)  Test: %d rows (%.2f%% churn)",
      len(X_train), y_train.mean() * 100,
      len(X_test), y_test.mean() * 100,
  )
  return X_train, y_train, X_test, y_test


In [ ]:
reg_time = pd.to_datetime(df["registration_init_time"])
df_proc, preprocessing_config = preprocess(df)
X_train, y_train, X_test, y_test = split(df_proc, reg_time)

In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 804315 entries, 4313 to 1082189
Data columns (total 18 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   city                804315 non-null  Int64  
 1   bd                  804315 non-null  Int64  
 2   gender              804315 non-null  int64  
 3   registered_via      804315 non-null  Int64  
 4   total_transactions  804315 non-null  Int64  
 5   total_amount_paid   804315 non-null  Int64  
 6   avg_amount_paid     804315 non-null  float64
 7   auto_renew_count    804315 non-null  Int64  
 8   cancel_count        804315 non-null  Int64  
 9   total_log_days      804315 non-null  Int64  
 10  total_secs          804315 non-null  float64
 11  avg_daily_secs      804315 non-null  float64
 12  total_num_25        804315 non-null  Int64  
 13  total_num_50        804315 non-null  Int64  
 14  total_num_75        804315 non-null  Int64  
 15  total_num_985       804315 non-null

In [ ]:
y_train.info()

<class 'pandas.core.series.Series'>
Index: 804315 entries, 4313 to 1082189
Series name: is_churn
Non-Null Count   Dtype
--------------   -----
804315 non-null  Int64
dtypes: Int64(1)
memory usage: 13.0 MB


# Train, evaluate

In [ ]:
def calculate_metrics(y_true, y_pred_proba, y_pred_class=None):
    # AUC ROC
    auc_roc = roc_auc_score(y_true, y_pred_proba)

    # Average Precision (AUC PR)
    auc_pr = average_precision_score(y_true, y_pred_proba)

    # Log Loss
    logloss = log_loss(y_true, y_pred_proba)

    # Threshold
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_pred_proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    best_threshold = thresholds[np.argmax(f1_scores[:-1])] if len(thresholds) > 0 else 0.5

    if y_pred_class is None:
        y_pred_class = (y_pred_proba >= best_threshold).astype(int)

    f1 = f1_score(y_true, y_pred_class)
    precision = precision_score(y_true, y_pred_class)
    recall = recall_score(y_true, y_pred_class)

    metrics = {
        "auc_roc": auc_roc,
        "auc_pr": auc_pr,
        "log_loss": logloss,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "best_threshold": best_threshold
    }

    return metrics

def train_and_evaluate_model(model, model_name, X_train, y_train, X_test, y_test):
    print(f"\n{'='*80}")
    print(f"Training {model_name}...")
    print(f"{'='*80}")

    start_time = time.time()

    model.fit(X_train, y_train)

    train_time = time.time() - start_time

    y_train_pred_proba = model.predict_proba(X_train)[:, 1]
    y_train_pred_class = model.predict(X_train)

    y_test_pred_proba = model.predict_proba(X_test)[:, 1]
    y_test_pred_class = model.predict(X_test)

    train_metrics = calculate_metrics(y_train, y_train_pred_proba, y_train_pred_class)
    test_metrics = calculate_metrics(y_test, y_test_pred_proba, y_test_pred_class)

    print(f"\n{model_name} - Training Time: {train_time:.2f} seconds")
    print(f"\n--- TRAIN SET METRICS ---")
    for metric, value in train_metrics.items():
        print(f"{metric:15s}: {value:.6f}")

    print(f"\n--- TEST SET METRICS ---")
    for metric, value in test_metrics.items():
        print(f"{metric:15s}: {value:.6f}")

    print(f"\n--- Classification Report on TEST SET ---")
    print(classification_report(y_test, y_test_pred_class, target_names=['No Churn', 'Churn']))

    return {
        'model_name': model_name,
        'train_time': train_time,
        'train_metrics': train_metrics,
        'test_metrics': test_metrics,
        'model': model
    }

models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        max_iter=1000,
        class_weight='balanced',
        n_jobs=-1
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=100,
        min_samples_leaf=50,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1,
        verbose=1
    ),

    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        min_samples_split=100,
        min_samples_leaf=50,
        random_state=42,
        verbose=1
    ),

    'Neural Network (MLP)': MLPClassifier(
        hidden_layer_sizes=(100, 50, 25),
        activation='relu',
        solver='adam',
        alpha=0.0001,
        batch_size=256,
        learning_rate='adaptive',
        max_iter=100,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        verbose=True
    )
}

results = {}


In [ ]:
result = train_and_evaluate_model(
        models['Logistic Regression'], 'Logistic Regression',
        X_train, y_train,
        X_test, y_test
    ),


Training Logistic Regression...

Logistic Regression - Training Time: 3.59 seconds

--- TRAIN SET METRICS ---
auc_roc        : 0.461053
auc_pr         : 0.097195
log_loss       : 0.693087
f1             : 0.129164
precision      : 0.091557
recall         : 0.219203
best_threshold : 0.494701

--- TEST SET METRICS ---
auc_roc        : 0.507971
auc_pr         : 0.116303
log_loss       : 0.693147
f1             : 0.045721
precision      : 0.306138
recall         : 0.024706
best_threshold : 0.500000

--- Classification Report on TEST SET ---
              precision    recall  f1-score   support

    No Churn       0.90      0.99      0.94    140561
       Churn       0.31      0.02      0.05     16555

    accuracy                           0.89    157116
   macro avg       0.60      0.51      0.49    157116
weighted avg       0.83      0.89      0.85    157116



In [ ]:
result = train_and_evaluate_model(
        models['Random Forest'], 'Random Forest',
        X_train, y_train,
        X_test, y_test
    ),


Training Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  3.4min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    3.9s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    7.0s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    2.7s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    6.1s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.4s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    0.9s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.6s
[Parallel(n_


Random Forest - Training Time: 207.30 seconds

--- TRAIN SET METRICS ---
auc_roc        : 0.872578
auc_pr         : 0.539106
log_loss       : 0.432839
f1             : 0.499754
precision      : 0.379288
recall         : 0.732358
best_threshold : 0.730118

--- TEST SET METRICS ---
auc_roc        : 0.882334
auc_pr         : 0.485364
log_loss       : 0.489821
f1             : 0.494518
precision      : 0.348011
recall         : 0.854062
best_threshold : 0.703243

--- Classification Report on TEST SET ---
              precision    recall  f1-score   support

    No Churn       0.98      0.81      0.89    140561
       Churn       0.35      0.85      0.49     16555

    accuracy                           0.82    157116
   macro avg       0.66      0.83      0.69    157116
weighted avg       0.91      0.82      0.85    157116



In [ ]:
result = train_and_evaluate_model(
        models['Gradient Boosting'], 'Gradient Boosting',
        X_train, y_train,
        X_test, y_test
    ),


Training Gradient Boosting...
      Iter       Train Loss   Remaining Time 
         1           0.6267            8.81m
         2           0.5994            9.19m
         3           0.5800            8.97m
         4           0.5652           10.78m
         5           0.5535           10.71m
         6           0.5442           10.84m
         7           0.5363           10.99m
         8           0.5299           10.86m
         9           0.5246           10.59m
        10           0.5203           10.48m
        20           0.4983            9.69m
        30           0.4915            7.82m
        40           0.4888            6.45m
        50           0.4870            5.30m
        60           0.4855            4.16m
        70           0.4841            3.07m
        80           0.4830            2.02m
        90           0.4820            1.00m
       100           0.4812            0.00s

Gradient Boosting - Training Time: 595.53 seconds

--- TRAIN SET ME

In [ ]:
result = train_and_evaluate_model(
        models['Neural Network (MLP)'], 'Neural Network (MLP)',
        X_train, y_train,
        X_test, y_test
    ),


Training Neural Network (MLP)...
Iteration 1, loss = 6.59455125
Validation score: 0.792595
Iteration 2, loss = 5.84335150
Validation score: 0.864581
Iteration 3, loss = 6.01952498
Validation score: 0.874229
Iteration 4, loss = 4.99572536
Validation score: 0.888378
Iteration 5, loss = 4.88573064
Validation score: 0.877723
Iteration 6, loss = 8.99119114
Validation score: 0.793888
Iteration 7, loss = 7.27048472
Validation score: 0.815235
Iteration 8, loss = 11.66997146
Validation score: 0.647143
Iteration 9, loss = 8.03544213
Validation score: 0.702432
Iteration 10, loss = 8.66799779
Validation score: 0.846765
Iteration 11, loss = 5.15409533
Validation score: 0.893649
Iteration 12, loss = 5.67931115
Validation score: 0.849475
Iteration 13, loss = 4.79123343
Validation score: 0.833785
Iteration 14, loss = 5.82026277
Validation score: 0.773896
Iteration 15, loss = 4.40972353
Validation score: 0.873346
Iteration 16, loss = 5.25915105
Validation score: 0.561766
Iteration 17, loss = 6.0413639